In [1]:
# ==========================================
# Section 1 : Install Required Libraries
# ==========================================

!pip -q install pypdf
!pip -q install faiss-cpu
!pip -q install sentence-transformers
!pip -q install transformers
!pip -q install torch
!pip -q install accelerate
!pip -q install langchain-text-splitters
!pip -q install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# Section 2 : Import Required Libraries

This section imports all the libraries required throughout the project.

These libraries help us perform:

- PDF Reading
- Text Chunking
- Embedding Generation
- Vector Database Creation
- Retrieval-Augmented Generation (RAG)
- Text Generation using FLAN-T5

In [2]:
# ==========================================
# Section 2 : Import Required Libraries
# ==========================================

import numpy as np
import faiss
import torch

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

print("✅ All Libraries Imported Successfully")

/tmp/ipykernel_1307/571708633.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ All Libraries Imported Successfully


# Section 3 : Upload and Read PDF

In this section, we upload a PDF document and convert it into LangChain Document objects.

Each page of the PDF becomes a separate Document object containing:
- page_content : Extracted text
- metadata : Source file and page number

In [4]:
# ==========================================
# Section 3 : Upload PDF
# ==========================================

def upload_pdf():
    uploaded = files.upload()

    pdf_path = list(uploaded.keys())[0]

    print(f"✅ Uploaded File : {pdf_path}")

    return pdf_path

In [5]:
# Upload PDF

pdf_path = upload_pdf()

Saving Engineering Intern - AI - ML Job Description (1).pdf to Engineering Intern - AI - ML Job Description (1).pdf
✅ Uploaded File : Engineering Intern - AI - ML Job Description (1).pdf


In [6]:
# ==========================================
# Section 3 : Read PDF
# ==========================================

def load_pdf(pdf_path):

    loader = PyPDFLoader(pdf_path)

    documents = loader.load()

    print(f"✅ Total Pages : {len(documents)}")

    return documents

In [7]:
documents = load_pdf(pdf_path)

✅ Total Pages : 4


In [8]:
print(documents[0].page_content[:1000])

Engineering Intern AI / ML - Job Description  
Who We Are: 
Ivanti manages, automates, and protects data and technology to empower continuous 
innovation. Our AI-powered platform, Ivanti Neurons, brings IT and Security teams 
together around a single, trusted system of record enabling smarter decisions, greater 
automation, and stronger security outcomes. 
At the core of our strategy is Autonomous Endpoint Management (AEM), which helps 
organisations move beyond manual, reactive approaches to IT and security. By 
combining authoritative data, automation, and agentic AI, Ivanti Neurons enables 
proactive, resilient, and scalable endpoint management helping customers focus on 
outcomes, not tools. 
Headquartered in the U.S. with a truly global footprint, Ivanti serves 34,000 customers 
across 149 countries, delivering mission-critical IT security, asset, and service 
management capabilities. 
Our award-winning software portfolio includes: 
• Ivanti Neurons Platform 
• Enterprise Service 

# Section 4 : Document Chunking

Large Language Models (LLMs) cannot efficiently process an entire PDF at once due to
context window limitations.

Instead, the document is divided into smaller overlapping chunks.

Why?

- Faster Retrieval
- Better Semantic Search
- Lower Memory Usage
- Improved Answer Quality

We use RecursiveCharacterTextSplitter to divide the document into manageable chunks.

In [9]:
# ==========================================
# Section 4 : Chunk Documents
# ==========================================

def chunk_documents(documents,
                    chunk_size=500,
                    chunk_overlap=100):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = text_splitter.split_documents(documents)

    print(f"✅ Total Chunks Created : {len(chunks)}")

    return chunks

In [10]:
chunks = chunk_documents(documents)

✅ Total Chunks Created : 15


In [11]:
print(chunks[0].page_content)

Engineering Intern AI / ML - Job Description  
Who We Are: 
Ivanti manages, automates, and protects data and technology to empower continuous 
innovation. Our AI-powered platform, Ivanti Neurons, brings IT and Security teams 
together around a single, trusted system of record enabling smarter decisions, greater 
automation, and stronger security outcomes. 
At the core of our strategy is Autonomous Endpoint Management (AEM), which helps


In [12]:
print(chunks[0].metadata)

{'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-06-26T13:47:41+00:00', 'author': 'Kanthasekar Pandian', 'moddate': '2026-06-26T13:47:41+00:00', 'source': 'Engineering Intern - AI - ML Job Description (1).pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


# Section 5 : Generate Text Embeddings

The chunks created in the previous section are still plain text.

To perform semantic search, we convert each chunk into a dense numerical vector
called an embedding.

We use the SentenceTransformer model:

**all-MiniLM-L6-v2**

Features:
- 384-dimensional embeddings
- Fast inference
- Lightweight
- Excellent semantic understanding
- Widely used in Retrieval-Augmented Generation (RAG)

In [13]:
# ==========================================
# Section 5 : Load Embedding Model
# ==========================================

def load_embedding_model():

    print("Loading embedding model...")

    model = SentenceTransformer("all-MiniLM-L6-v2")

    print("✅ Embedding Model Loaded Successfully")

    return model

In [14]:
embedding_model = load_embedding_model()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding Model Loaded Successfully


In [15]:
# ==========================================
# Section 5 : Create Embeddings
# ==========================================

def create_embeddings(chunks, embedding_model):

    chunk_texts = [chunk.page_content for chunk in chunks]

    embeddings = embedding_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    print(f"✅ Embeddings Created : {len(embeddings)}")

    return embeddings

In [16]:
embeddings = create_embeddings(
    chunks,
    embedding_model
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings Created : 15


In [17]:
print("Number of Embeddings :", len(embeddings))
print("Embedding Dimension :", embeddings.shape[1])

Number of Embeddings : 15
Embedding Dimension : 384


In [18]:
print(embeddings[0][:10])

[-0.03487396 -0.06007228  0.0198105  -0.03347657  0.02285809 -0.04583522
  0.02012909  0.02557558 -0.01564808 -0.01772955]


# Section 6 : Build FAISS Vector Database

Embeddings by themselves are simply vectors.

To efficiently search among thousands (or even millions) of vectors,
we use **FAISS (Facebook AI Similarity Search)**.

FAISS stores the embeddings and performs **Nearest Neighbor Search**
to retrieve the chunks that are semantically closest to a user's question.

In this project we use:

**IndexFlatL2**

which performs exact nearest-neighbor search using Euclidean (L2) Distance.

In [19]:
# ==========================================
# Section 6 : Build FAISS Index
# ==========================================

def build_faiss_index(embeddings):

    embeddings = embeddings.astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    print("✅ FAISS Index Created Successfully")
    print(f"Total Vectors Stored : {index.ntotal}")

    return index

In [20]:
index = build_faiss_index(embeddings)

✅ FAISS Index Created Successfully
Total Vectors Stored : 15


In [21]:
print(index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7e5e8e3a0d30> >


In [22]:
print("Vectors in Database :", index.ntotal)

Vectors in Database : 15


# Section 7 : Load FLAN-T5 Base

After retrieving the most relevant document chunks using FAISS,
we need a Large Language Model (LLM) to generate the final answer.

We use **FLAN-T5 Base**, an instruction-tuned open-source language model developed by Google.

Why FLAN-T5?

- Open Source
- No API Key Required
- Runs locally in Google Colab
- Supports Question Answering
- Lightweight compared to larger LLMs

In [59]:
def load_llm():

    print("Loading FLAN-T5 Base...")

    model_name = "google/flan-t5-large"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if torch.cuda.is_available():
        model = model.to("cuda")

    print("✅ FLAN-T5 Loaded Successfully")

    return tokenizer, model

In [60]:
tokenizer, model = load_llm()

Loading FLAN-T5 Base...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ FLAN-T5 Loaded Successfully


In [42]:
def generate_text(prompt):

    # Convert text into tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Move tensors to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=False
    )

    # Convert tokens back to text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [43]:
prompt = "Question: What is Artificial Intelligence?"

answer = generate_text(prompt)

print(answer)

artificial intelligence is development of artificial intelligence


# Section 8 : Retrieve Relevant Chunks

When the user asks a question:

1. Convert the question into an embedding.
2. Search the FAISS index.
3. Retrieve the most relevant chunks.
4. Send only those chunks to the LLM.

This is the Retrieval step of Retrieval-Augmented Generation (RAG).

In [44]:
# ==========================================
# Section 8 : Retrieve Relevant Chunks
# ==========================================

def retrieve_context(query,
                     embedding_model,
                     index,
                     chunks,
                     top_k=3):

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    # Collect retrieved chunks
    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx].page_content)

    context = "\n\n".join(retrieved_chunks)

    return context

In [45]:
query = "What is this PDF about?"

context = retrieve_context(
    query,
    embedding_model,
    index,
    chunks
)

print(context)

cases, data processing and prototype development. This internship offers hands-on 
exposure to real-world engineering practices and emerging AI technologies in a global 
software environment. 
 
What You Will Do

Own It: Debate, decide, commit. We take responsibility for our actions and decisions—
demonstrating commitment, integrity, high standards, and a drive for excellence in 
everything we do. 
About the Role 
We are seeking a highly motivated and curious AI / Machine Learning Intern to join 
our engineering team. This role is ideal for someone with foundational exposure to 
Artificial Intelligence through academics’ study, personal projects, or prior internships

Engineering Intern AI / ML - Job Description  
Who We Are: 
Ivanti manages, automates, and protects data and technology to empower continuous 
innovation. Our AI-powered platform, Ivanti Neurons, brings IT and Security teams 
together around a single, trusted system of record enabling smarter decisions, greater 
automatio

# Section 9 : Prompt Construction

The retrieved chunks are combined into a prompt.

The prompt contains:

1. Retrieved Context
2. User Question

The prompt is then passed to FLAN-T5 to generate a context-aware answer.

In [50]:
# ==========================================
# Section 9 : Build Prompt
# ==========================================

def build_prompt(context, query):

    return f"""
Answer the question based on the context below.

Context:
{context}

Question:
{query}

Give a concise answer.
"""

In [51]:
prompt = build_prompt(
    context,
    query
)

print(prompt)


Answer the question based on the context below.

Context:
cases, data processing and prototype development. This internship offers hands-on 
exposure to real-world engineering practices and emerging AI technologies in a global 
software environment. 
 
What You Will Do

Own It: Debate, decide, commit. We take responsibility for our actions and decisions—
demonstrating commitment, integrity, high standards, and a drive for excellence in 
everything we do. 
About the Role 
We are seeking a highly motivated and curious AI / Machine Learning Intern to join 
our engineering team. This role is ideal for someone with foundational exposure to 
Artificial Intelligence through academics’ study, personal projects, or prior internships

Engineering Intern AI / ML - Job Description  
Who We Are: 
Ivanti manages, automates, and protects data and technology to empower continuous 
innovation. Our AI-powered platform, Ivanti Neurons, brings IT and Security teams 
together around a single, trusted syst

In [54]:
def generate_answer(prompt):

    inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    num_beams=5,
    early_stopping=True,
    no_repeat_ngram_size=2
)

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [55]:
final_answer = generate_answer(prompt)

print(final_answer)

AI / Machine Learning
